# Travel Companion AI (Google Colab Edition)

This notebook provides:
- NLP flight intent extraction
- Optional **SerpApi Google Flights** live results
- Automatic **Synthetic Demo** fallback when no key is configured
- ipywidgets-based interactive chat UI

In [ ]:
# Install required packages
!pip -q install requests ipywidgets spacy nltk

In [ ]:
# Optional: install OpenAI SDK only when needed
USE_OPENAI_PARSER = False  # set True only if you want optional LLM structured extraction
if USE_OPENAI_PARSER:
    !pip -q install openai

In [ ]:
import re
import random
from dataclasses import dataclass, asdict, field
from datetime import datetime, timedelta
from typing import List, Optional, Dict, Any

import requests
import nltk
import spacy
from IPython.display import display, HTML
import ipywidgets as widgets

# Colab secrets
try:
    from google.colab import userdata
    SERPAPI_API_KEY = userdata.get("SERPAPI_API_KEY")
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    SERPAPI_API_KEY = None
    OPENAI_API_KEY = None

nltk.download('punkt', quiet=True)

In [ ]:
# Load spaCy model (fallback to blank English if model is unavailable)
try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    !python -m spacy download en_core_web_sm -q
    try:
        nlp = spacy.load("en_core_web_sm")
    except Exception:
        nlp = spacy.blank("en")

In [ ]:
@dataclass
class TravelQuery:
    origin_city: Optional[str] = None
    destination_city: Optional[str] = None
    departure_date: Optional[str] = None  # YYYY-MM-DD
    return_date: Optional[str] = None     # YYYY-MM-DD
    num_passengers: int = 1
    cabin_class: str = "economy"
    max_price: Optional[float] = None
    preferred_airline: Optional[str] = None
    direct_flight_only: bool = False
    trip_type: str = "one-way"

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

@dataclass
class FlightResult:
    airline: str
    flight_number: Optional[str]
    origin_city: str
    origin_code: Optional[str]
    destination_city: str
    destination_code: Optional[str]
    departure_time: str
    arrival_time: str
    duration: str
    price: float
    currency: str
    cabin_class: str
    stops: int
    booking_source: Optional[str]
    is_live_result: bool = False

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

@dataclass
class SearchResponse:
    label: str
    query: TravelQuery
    flights: List[FlightResult] = field(default_factory=list)
    errors: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "label": self.label,
            "query": self.query.to_dict(),
            "flights": [f.to_dict() for f in self.flights],
            "errors": self.errors,
        }

In [ ]:
class NLPService:
    city_pattern = re.compile(r"from\s+([A-Za-z\s]+?)\s+to\s+([A-Za-z\s]+?)(?:\s|$)", re.IGNORECASE)

    def parse(self, text: str) -> TravelQuery:
        # 1) optional OpenAI extraction
        if OPENAI_API_KEY:
            oq = self._parse_with_openai(text)
            if oq:
                return oq
        # 2) spaCy + regex
        q = self._parse_with_rules(text)
        # 3) simple fallback
        if not q.origin_city or not q.destination_city:
            q = self._parse_simple_fallback(text, q)
        return q

    def _normalize_date(self, phrase: str) -> Optional[str]:
        today = datetime.utcnow().date()
        p = phrase.lower()
        if "tomorrow" in p:
            return str(today + timedelta(days=1))
        weekdays = ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
        for i,w in enumerate(weekdays):
            if f"next {w}" in p:
                days_ahead = (i - today.weekday()) % 7
                days_ahead = 7 if days_ahead == 0 else days_ahead
                return str(today + timedelta(days=days_ahead))
        return None

    def _parse_with_openai(self, text: str) -> Optional[TravelQuery]:
        try:
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            prompt = f"""Extract flight fields from: {text}
Return JSON with keys origin_city,destination_city,departure_date,return_date,num_passengers,cabin_class,max_price,preferred_airline,direct_flight_only,trip_type"""
            r = client.chat.completions.create(model="gpt-4o-mini",messages=[{"role":"user","content":prompt}],temperature=0)
            import json
            obj = json.loads(r.choices[0].message.content)
            return TravelQuery(**obj)
        except Exception:
            return None

    def _parse_with_rules(self, text: str) -> TravelQuery:
        q = TravelQuery()
        t = text.strip()
        m = self.city_pattern.search(t)
        if m:
            q.origin_city = m.group(1).strip()
            q.destination_city = m.group(2).strip()
        dp = self._normalize_date(t)
        if dp:
            q.departure_date = dp
        mrange = re.search(r"from\s+([A-Za-z]+\s+\d{1,2})\s+to\s+([A-Za-z]+\s+\d{1,2})", t, re.IGNORECASE)
        if mrange:
            year = datetime.utcnow().year
            for attr, val in [("departure_date",mrange.group(1)),("return_date",mrange.group(2))]:
                try:
                    setattr(q, attr, datetime.strptime(f"{val} {year}", "%B %d %Y").date().isoformat())
                except Exception:
                    pass
        if re.search(r"round\s*-?trip|return", t, re.IGNORECASE):
            q.trip_type = "round-trip"
        if re.search(r"business", t, re.IGNORECASE):
            q.cabin_class = "business"
        elif re.search(r"first class|first", t, re.IGNORECASE):
            q.cabin_class = "first"
        if re.search(r"direct flights? only|nonstop", t, re.IGNORECASE):
            q.direct_flight_only = True
        mp = re.search(r"under\s+(\d+)", t, re.IGNORECASE)
        if mp:
            q.max_price = float(mp.group(1))
        pax = re.search(r"for\s+(\d+)\s+passengers?", t, re.IGNORECASE)
        if pax:
            q.num_passengers = int(pax.group(1))
        air = re.search(r"with\s+([A-Za-z\s]+?)\s+(?:airlines?|airways)", t, re.IGNORECASE)
        if air:
            q.preferred_airline = air.group(1).strip()
        return q

    def _parse_simple_fallback(self, text: str, q: TravelQuery) -> TravelQuery:
        parts = re.findall(r"[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*", text)
        if len(parts) >= 2:
            q.origin_city = q.origin_city or parts[0]
            q.destination_city = q.destination_city or parts[1]
        if not q.departure_date:
            q.departure_date = str(datetime.utcnow().date() + timedelta(days=7))
        return q

In [ ]:
class SerpApiFlightService:
    endpoint = "https://serpapi.com/search.json"

    def __init__(self, api_key: Optional[str]):
        self.api_key = api_key

    def search(self, query: TravelQuery) -> SearchResponse:
        if not self.api_key:
            return SearchResponse(label="Synthetic Demo Results", query=query, flights=[])
        params = {
            "engine": "google_flights",
            "api_key": self.api_key,
            "departure_id": query.origin_city or "",
            "arrival_id": query.destination_city or "",
            "outbound_date": query.departure_date or str(datetime.utcnow().date()+timedelta(days=7)),
            "type": "2" if query.trip_type == "round-trip" else "1",
            "adults": query.num_passengers,
            "travel_class": {"economy":"1","premium_economy":"2","business":"3","first":"4"}.get(query.cabin_class.lower(),"1"),
            "stops": "0" if query.direct_flight_only else "1",
            "currency": "USD",
            "hl": "en",
            "gl": "us",
        }
        if query.return_date:
            params["return_date"] = query.return_date
        errors=[]
        flights=[]
        try:
            res = requests.get(self.endpoint, params=params, timeout=20)
            res.raise_for_status()
            data = res.json()
            for item in data.get("best_flights", []) + data.get("other_flights", []):
                flights.append(self._parse_flight(item, query))
        except Exception as e:
            errors.append(str(e))
        flights = [f for f in flights if f is not None]
        if query.max_price is not None:
            flights = [f for f in flights if f.price <= query.max_price]
        flights.sort(key=lambda x: x.price)
        return SearchResponse(label="Live API Results" if flights else "Synthetic Demo Results", query=query, flights=flights, errors=errors)

    def _parse_flight(self, item: dict, q: TravelQuery):
        try:
            legs = item.get("flights", [])
            first = legs[0] if legs else {}
            last = legs[-1] if legs else {}
            return FlightResult(
                airline=first.get("airline", "Unknown Airline"),
                flight_number=first.get("flight_number"),
                origin_city=q.origin_city or first.get("departure_airport", {}).get("name", "Unknown"),
                origin_code=first.get("departure_airport", {}).get("id"),
                destination_city=q.destination_city or last.get("arrival_airport", {}).get("name", "Unknown"),
                destination_code=last.get("arrival_airport", {}).get("id"),
                departure_time=first.get("departure_airport", {}).get("time", "N/A"),
                arrival_time=last.get("arrival_airport", {}).get("time", "N/A"),
                duration=item.get("total_duration", "N/A"),
                price=float(str(item.get("price", 0)).replace('$','').replace(',','') or 0),
                currency="USD",
                cabin_class=q.cabin_class,
                stops=len(legs)-1 if legs else 0,
                booking_source="Google Flights via SerpApi",
                is_live_result=True,
            )
        except Exception:
            return None

In [ ]:
class SyntheticFlightService:
    airlines = ["SkyJet", "AeroNova", "FlyWorld", "CloudAir", "BlueWings"]

    def search(self, query: TravelQuery, n: int = 6) -> SearchResponse:
        dep = query.departure_date or str(datetime.utcnow().date()+timedelta(days=7))
        flights=[]
        base = random.randint(120, 480)
        for i in range(n):
            price = max(60, base + random.randint(-60, 220))
            if query.max_price and price > query.max_price:
                continue
            stops = 0 if query.direct_flight_only else random.choice([0,1])
            flights.append(FlightResult(
                airline=query.preferred_airline or random.choice(self.airlines),
                flight_number=f"TC{random.randint(100,999)}",
                origin_city=query.origin_city or "Origin",
                origin_code=None,
                destination_city=query.destination_city or "Destination",
                destination_code=None,
                departure_time=f"{dep} {random.randint(5,22):02d}:{random.choice([0,15,30,45]):02d}",
                arrival_time=f"{dep} {random.randint(6,23):02d}:{random.choice([0,15,30,45]):02d}",
                duration=f"{random.randint(2,14)}h {random.choice([0,15,30,45])}m",
                price=float(price),
                currency="USD",
                cabin_class=query.cabin_class,
                stops=stops,
                booking_source="Synthetic Engine",
                is_live_result=False,
            ))
        flights.sort(key=lambda x: x.price)
        return SearchResponse(label="Synthetic Demo Results", query=query, flights=flights)

In [ ]:
nlp_service = NLPService()
serp_service = SerpApiFlightService(SERPAPI_API_KEY)
synth_service = SyntheticFlightService()

def run_search(user_text: str) -> SearchResponse:
    query = nlp_service.parse(user_text)
    live = serp_service.search(query)
    if live.flights:
        return live
    fallback = synth_service.search(query)
    fallback.errors.extend(live.errors)
    return fallback

def render_response(resp: SearchResponse):
    q = resp.query.to_dict()
    qhtml = "".join([f"<li><b>{k}</b>: {v}</li>" for k,v in q.items()])
    cards = []
    for f in resp.flights[:10]:
        cards.append(f"""
        <div style='border:1px solid #ddd;border-radius:10px;padding:10px;margin:8px 0;'>
          <b>{f.airline} {f.flight_number or ''}</b> | ${f.price:.0f} {f.currency}<br>
          {f.origin_city} → {f.destination_city}<br>
          Depart: {f.departure_time} | Arrive: {f.arrival_time}<br>
          Duration: {f.duration} | Stops: {f.stops} | Cabin: {f.cabin_class}
        </div>
        """)
    display(HTML(f"""
    <h3>{resp.label}</h3>
    <h4>Extracted Travel Understanding</h4><ul>{qhtml}</ul>
    <h4>Flight Options</h4>{''.join(cards) or '<p>No flights found.</p>'}
    {('<p><b>Notes:</b> ' + '; '.join(resp.errors) + '</p>') if resp.errors else ''}
    """))

input_box = widgets.Text(placeholder="e.g., direct flights from London to Tokyo tomorrow", layout=widgets.Layout(width='75%'))
search_btn = widgets.Button(description="Search Flights", button_style='primary')
out = widgets.Output()

def on_search(_):
    with out:
        out.clear_output()
        render_response(run_search(input_box.value))

search_btn.on_click(on_search)
display(widgets.HBox([input_box, search_btn]), out)

In [ ]:
# Optional: download this notebook file from Colab
try:
    from google.colab import files
    files.download('Travel_Companion_AI_Colab.ipynb')
except Exception:
    print('Run this cell inside Google Colab to download the notebook file.')
